In [ ]:
import os
import sys
import numpy as np
import glob
import torch
from PIL import Image

from google.colab import drive
drive.mount('/content/drive')

sys.path.insert(0, '/content/drive/MyDrive/video_surgery')

from src import setup, config

setup.prep_env(config.COLAB_ROOT)
setup.download_models()
sys.path.insert(0, config.COLAB_GROUNDED_SAM_DIR)
sys.path.insert(0, f"{config.COLAB_RAFT_DIR}/core")

print('\nRestart Session to continue')

In [ ]:
from src import helpers

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
predictor = helpers.load_sam2_video(device)

In [ ]:
anchor_mask = np.load(config.ANCHOR_MASK_PATH)
print(f"Anchor mask loaded - shape: {anchor_mask.shape}, pixels: {anchor_mask.sum()}")

In [ ]:
frame_paths = sorted(config.FRAMES.glob('*.jpg'))
print(f"Total frames to propagate: {len(frame_paths)}")

existing = [f for f in frame_paths if (config.MASKS / f"{f.stem}.npy").exists()]
if len(existing) == len(frame_paths):
    print('All frames are processed')
else:
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):

        inference_state = predictor.init_state(video_path=str(config.FRAMES))

        predictor.add_new_mask(
            inference_state,
            frame_idx=0,
            obj_id=1,
            mask=anchor_mask
        )

        for frame_idx, obj_ids, mask_logits in predictor.propagate_in_video(inference_state):
            mask = (mask_logits[0, 0] > 0.0).cpu().numpy().astype(np.uint8)

            out_path = config.MASKS / f"{frame_idx}.npy"
            if not out_path.exists():
                np.save(out_path, mask)

            if frame_idx % 5 == 0:
                print(f"  Frame {frame_idx:03d} - mask pixels: {mask.sum()}")

    print(f"\nPropagation complete, masks saved to {config.MASKS}")

In [ ]:
import cv2
import matplotlib.pyplot as plt

gt_paths   = sorted(config.GT_DIR.glob('*.png'), key=lambda p: int(p.stem))
prop_paths = sorted(config.MASKS.glob('*.npy'), key=lambda p: int(p.stem))


def compute_iou(mask_a, mask_b):
    intersection = (mask_a & mask_b).sum()
    union        = (mask_a | mask_b).sum()
    return intersection / (union + 1e-6)

ious        = []
flagged     = []

for i, (prop_path, gt_path) in enumerate(zip(prop_paths, gt_paths)):
    prop_mask = np.load(prop_path)

    gt_raw  = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)
    gt_mask = (gt_raw > 0).astype(np.uint8)

    # Resize gt to match propagated mask if needed
    if gt_mask.shape != prop_mask.shape:
        gt_mask = cv2.resize(
            gt_mask, (prop_mask.shape[1], prop_mask.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

    iou = compute_iou(prop_mask, gt_mask)
    ious.append(iou)

    if iou < 0.7:
        flagged.append((i, iou))

print(f"Mean IoU:   {np.mean(ious):.4f}")
print(f"Min IoU:    {np.min(ious):.4f}  (frame {np.argmin(ious)})")
print(f"Max IoU:    {np.max(ious):.4f}")
print(f"Flagged frames (IoU < 0.7): {flagged if flagged else 'none'}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ious, linewidth=2, color='steelblue', label='IoU vs GT')
ax.axhline(0.7, color='red', linestyle='--', linewidth=1, label='threshold (0.7)')
ax.fill_between(range(len(ious)), ious, 0.7,
                where=[v < 0.7 for v in ious],
                color='red', alpha=0.2, label='below threshold')
ax.set_xlabel('Frame index')
ax.set_ylabel('IoU')
ax.set_title(f"SAM 2 mask propagation - {config.SEQ}")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
sample_indices = np.linspace(0, len(prop_paths) - 1, 5, dtype=int)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for col, idx in enumerate(sample_indices):
    frame_path = config.FRAMES / f"{idx}.jpg"
    mask_path  = config.MASKS / f"{idx}.npy"

    frame = cv2.cvtColor(cv2.imread(frame_path), cv2.COLOR_BGR2RGB)
    mask  = np.load(mask_path)

    overlay = frame.copy().astype(np.float32)
    overlay[mask == 1] = (
        overlay[mask == 1] * 0.5 + np.array([255, 0, 0]) * 0.5
    )
    overlay = overlay.astype(np.uint8)

    axes[0, col].imshow(frame)
    axes[0, col].set_title(f"Frame {idx}")
    axes[0, col].axis('off')

    axes[1, col].imshow(overlay)
    axes[1, col].set_title(f"IoU: {ious[idx]:.3f}")
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=12)
axes[1, 0].set_ylabel('Mask overlay', fontsize=12)

plt.suptitle(f"SAM 2 propagation - {config.SEQ}", fontsize=14)
plt.tight_layout()
